In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

In [ ]:
folder_path = r"/storage/alplakes_test/lucerne_100m_2025"
input_folder = os.path.join(folder_path, "outputs_swirl", "eddy_catalogues_lvl1")

output_folder = os.path.join(folder_path, "outputs_swirl", "eddy_statistics")
os.makedirs(output_folder, exist_ok=True)

# Import lvl1 catalogue

In [ ]:
lvl1_csv_path = os.path.join(input_folder, "lvl1_corr.csv")

In [ ]:
df_lvl1 = pd.read_csv(lvl1_csv_path)
df_lvl1 = df_lvl1.set_index('id', drop=False)
df_lvl1['date'] = pd.to_datetime(df_lvl1['date'])

In [ ]:
lake_mask = np.load(os.path.join(folder_path, "grid", "mask_lake.npy"))

In [ ]:
depths = pd.read_csv(os.path.join(folder_path, "grid", "depths.csv"))

# Number of eddies

In [ ]:
nb_eddy = df_lvl1.groupby(['date'])['id'].count()

In [ ]:
df_lvl1.groupby(['date'])['id'].count().iloc[7010]

In [ ]:
fig, ax = plt.subplots()
nb_eddy.plot()
plt.ylim(bottom=0)
plt.ylabel("Number of eddies")
plt.xlabel("")
plt.savefig(os.path.join(output_folder, "eddy_numbers.png"))

In [ ]:
def filter_by_depths(df, depth_min, depth_max):
    depth_filter = (
            (abs(df['depth_min_[m]']) >= abs(depth_max)) &
            (abs(df['depth_max_[m]']) <= abs(depth_min))
    )

    return df[depth_filter]

In [ ]:
depth_min = -10
depth_max = -0
df_lvl1_filtered_by_depth = filter_by_depths(df_lvl1, depth_min, depth_max)

In [ ]:
df_lvl1_filtered_by_depth[df_lvl1_filtered_by_depth['date']=='2025-05-12 03:30:00']

In [ ]:
nb_eddy_by_depth = df_lvl1_filtered_by_depth.groupby(['date'])['id'].count()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
nb_eddy_by_depth.plot()
plt.ylim(bottom=0)
plt.ylabel("Number of eddies")
plt.text(0.02, 0.98, f'{depth_min} to {depth_max}m', transform=plt.gca().transAxes, ha='left', va='top')
plt.xlabel("")
plt.savefig(os.path.join(output_folder, f"eddy_numbers_{depth_min}-{depth_max}m.png"))

In [ ]:
nb_eddy_by_depth.reset_index().to_csv(os.path.join(output_folder, f"eddy_numbers_{depth_min}-{depth_max}m.csv"))

# Surface statistics

In [ ]:
df_lvl1['surface_area_mean_[km2]'] = df_lvl1['surface_area_mean_[m2]'] / 1e6

In [ ]:
surface_timeserie = df_lvl1.groupby(['date'])['surface_area_mean_[km2]'].sum()

In [ ]:
fig, ax = plt.subplots()
surface_timeserie.plot()
plt.ylabel("Mean surface area [km$^2$]")
plt.xlabel('')
plt.savefig(os.path.join(output_folder, "surface_timeserie.png"))

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df_lvl1['surface_area_mean_[km2]'], bins=50, kde=True)
plt.xlabel("Surface area [km2]")
ax.xaxis.set_major_locator(MultipleLocator(1))
plt.savefig(os.path.join(output_folder, "surface_statistics.png"))

# Volume statistics

In [ ]:
df_lvl1.head()

In [ ]:
df_lvl1['volume_[km3]'] = df_lvl1['volume_[m3]'] / 1e9

In [ ]:
volume_timeserie = df_lvl1.groupby(['date'])['volume_[km3]'].sum()

In [ ]:
fig, ax = plt.subplots()
volume_timeserie.plot()
plt.ylabel("Volume [km$^3$]")
plt.xlabel('')
plt.savefig(os.path.join(output_folder, "volume_timeserie.png"))

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df_lvl1['volume_[km3]'], bins=50)
plt.xlabel("Volume [km$^3$]")
plt.savefig(os.path.join(output_folder, "volume_statistics.png"))

# Height statistics

In [ ]:
depths = pd.read_csv(os.path.join(folder_path, "grid", "depths.csv"))

In [ ]:
depths.head()

In [ ]:
df_lvl1['height_[m]'] = df_lvl1['depth_max_[m]'] - df_lvl1['depth_min_[m]']

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df_lvl1['height_[m]'], bins=50, kde=True)
plt.xlabel("Height [m]")
#ax.xaxis.set_major_locator(MultipleLocator(1))
plt.savefig(os.path.join(output_folder, "height_statistics.png"))